# Ablation Study: Independent BERT and RoBERTa Models
This notebook replicates the ablation study portion of the paper, where standalone BERT and RoBERTa models are evaluated independently before being compared against the Hybrid model.

In [ ]:
import pandas as pd
import numpy as np
import torch
import os
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, RobertaTokenizer
from torch.cuda.amp import autocast, GradScaler
from sklearn.metrics import classification_report, accuracy_score, f1_score
from tqdm.auto import tqdm
from collections import Counter
import random

from models import BERTClassifier, RoBERTaClassifier
from augmentation import RequirementAugmenter

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)


## 1. Load Data & Prepare Data Loaders
We'll reuse the exact same Multi-class (NFR) data and Augmentation process to keep the comparison fair.

In [ ]:
data_dir = '../data/processed'
train_df = pd.read_csv(f"{data_dir}/train.csv")
val_df = pd.read_csv(f"{data_dir}/val.csv")
test_df = pd.read_csv(f"{data_dir}/test.csv")

# Filter NFRs
train_nfr = train_df[train_df['label_binary'] == 'NFR'].copy().reset_index(drop=True)
val_nfr = val_df[val_df['label_binary'] == 'NFR'].copy().reset_index(drop=True)
test_nfr = test_df[test_df['label_binary'] == 'NFR'].copy().reset_index(drop=True)

unique_labels = sorted(train_nfr['sub_NFR'].unique())
label2id = {label: i for i, label in enumerate(unique_labels)}
id2label = {i: label for i, label in enumerate(unique_labels)}
num_classes = len(unique_labels)

train_nfr['label_id'] = train_nfr['sub_NFR'].map(label2id)
val_nfr['label_id'] = val_nfr['sub_NFR'].map(label2id)
test_nfr['label_id'] = test_nfr['sub_NFR'].map(label2id)

# --- Data Augmentation ---
aug = RequirementAugmenter()
class_counts = Counter(train_nfr['sub_NFR'])
max_count = max(class_counts.values())

augmented_rows = []
for label, count in class_counts.items():
    if count < max_count:
        num_to_generate = max_count - count
        existing_texts = train_nfr[train_nfr['sub_NFR'] == label]['text'].tolist()
        for _ in range(num_to_generate):
            base_text = random.choice(existing_texts)
            try:
                new_text = aug.augment(base_text, strategy='random')
            except:
                new_text = base_text
            augmented_rows.append({'text': new_text, 'sub_NFR': label, 'label_id': label2id[label]})

aug_df = pd.DataFrame(augmented_rows)
train_nfr_balanced = pd.concat([train_nfr, aug_df], ignore_index=True)


## 2. Shared Single-Model Dataset Class

In [ ]:
class SingleRequirementDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=32):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
        
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        
        enc = self.tokenizer(
            text, truncation=True, padding='max_length', max_length=self.max_length, return_tensors='pt'
        )
        return {
            'input_ids': enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'labels': torch.tensor(label, dtype=torch.long)
        }

def get_loaders(tokenizer):
    train_dataset = SingleRequirementDataset(train_nfr_balanced['text'].values, train_nfr_balanced['label_id'].values, tokenizer)
    val_dataset = SingleRequirementDataset(val_nfr['text'].values, val_nfr['label_id'].values, tokenizer)
    test_dataset = SingleRequirementDataset(test_nfr['text'].values, test_nfr['label_id'].values, tokenizer)
    return (
        DataLoader(train_dataset, batch_size=16, shuffle=True),
        DataLoader(val_dataset, batch_size=16, shuffle=False),
        DataLoader(test_dataset, batch_size=16, shuffle=False)
    )


## 3. Training Loop Helper

In [ ]:
def train_model(model, train_loader, val_loader, model_name):
    model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5, weight_decay=0.02)
    criterion = torch.nn.CrossEntropyLoss()
    # scaler removed for CPU
    
    epochs = 3
    accumulation_steps = 1
    best_val_f1 = 0.0
    patience = 2
    patience_counter = 0
    
    os.makedirs('../models/ablation', exist_ok=True)
    best_model_path = f'../models/ablation/{model_name}.pt'
    
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        optimizer.zero_grad()
        
        for i, batch in enumerate(tqdm(train_loader, desc=f"{model_name} Epoch {epoch+1}")):
            input_ids = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            if True: # autocast removed for CPU
                logits = model(input_ids, mask)
                loss = criterion(logits, labels) / accumulation_steps
                
            loss.backward()
            
            if (i + 1) % accumulation_steps == 0:
                optimizer.step()
                
                optimizer.zero_grad()
                
            total_loss += loss.item() * accumulation_steps
            
        # Validation
        model.eval()
        val_preds, val_labels = [], []
        with torch.no_grad():
            for batch in val_loader:
                if True: # autocast removed for CPU
                    logits = model(batch['input_ids'].to(device), batch['attention_mask'].to(device))
                preds = torch.argmax(logits, dim=1).cpu().numpy()
                val_preds.extend(preds)
                val_labels.extend(batch['labels'].numpy())
                
        val_f1 = f1_score(val_labels, val_preds, average='macro')
        print(f"{model_name} | Val F1: {val_f1:.4f}")
        
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            torch.save(model.state_dict(), best_model_path)
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print("Early stopping triggered!")
                break
                
    return best_model_path

def evaluate_model(model, test_loader, best_path):
    model.load_state_dict(torch.load(best_path))
    model.eval()
    test_preds, test_labels = [], []
    with torch.no_grad():
        for batch in test_loader:
            if True: # autocast removed for CPU
                logits = model(batch['input_ids'].to(device), batch['attention_mask'].to(device))
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            test_preds.extend(preds)
            test_labels.extend(batch['labels'].numpy())
    print(f"Accuracy: {accuracy_score(test_labels, test_preds):.4f}")
    print(classification_report(test_labels, test_preds, target_names=[id2label[i] for i in range(num_classes)]))


## 4. Run BERT Ablation

In [ ]:
bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
train_l, val_l, test_l = get_loaders(bert_tokenizer)

bert_model = BERTClassifier(num_classes=num_classes)
print("Training BERT...")
bert_path = train_model(bert_model, train_l, val_l, 'bert_standalone')
print("\n--- BERT Standalone Test Results ---")
evaluate_model(bert_model, test_l, bert_path)


## 5. Run RoBERTa Ablation

In [ ]:
roberta_tokenizer = RobertaTokenizer.from_pretrained('roberta-base')
train_l, val_l, test_l = get_loaders(roberta_tokenizer)

roberta_model = RoBERTaClassifier(num_classes=num_classes)
print("\nTraining RoBERTa...")
roberta_path = train_model(roberta_model, train_l, val_l, 'roberta_standalone')
print("\n--- RoBERTa Standalone Test Results ---")
evaluate_model(roberta_model, test_l, roberta_path)
